# UMAP exploration of passage and text embeddings

Visualise the structure of e5-large (1024-d) passage embeddings by projecting to 2D with UMAP, then colouring by genre / period / abstractness / annotation labels.

Uses `build_feature_matrix` and `load_genre_extras` from `largeliterarymodels.analysis`, plus `passage_abstractness_by_lang`-style joins via `abstraction.aggregate`.

Sections:
1. **Passage-level UMAP** on the ~1,700 form-annotated passages, coloured by genre, period, abstractness, and thesis annotations.
2. **Genre-faceted passage UMAP** (coloured by abstractness and by period).
3. **Text-level UMAP** (mean-pooled) for all 3K EN+FR texts — raw and language-centered versions.
4. **Genre-faceted text UMAP** for the top 20 tags.

In [ ]:
import clickhouse_connect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from largeliterarymodels.analysis import (
    build_feature_matrix, load_genre_extras,
    fetch_passage_embeddings, mean_pool_to_text, center_by_group,
)
from abstraction.analysis import attach_extras, run_umap
from abstraction.plotting import (
    plot_umap_overview, facet_by_genre, plot_text_umap_4panel,
    add_period_bin, PERIOD_COLORS, FORM_GENRE_COLORS,
)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 200

c = clickhouse_connect.get_client(host='localhost', port=8123, username='lltk', password='lltk')

All data-plumbing helpers now live in two modules:

- `largeliterarymodels.analysis` — `fetch_passage_embeddings`, `mean_pool_to_text`, `center_by_group` (task-agnostic embedding ops)
- `abstraction.analysis` — `attach_extras`, `run_umap` (pipeline glue specific to this thesis)
- `abstraction.plotting` — `plot_umap_overview`, `facet_by_genre`, `plot_text_umap_4panel`, `add_period_bin` (figure builders)

## 1. Passage-level UMAP (form-annotated subset)

Build the annotation feature matrix (same one used in notebook 12), attach genre tags, pull embeddings, UMAP.

In [ ]:
features, groups = build_feature_matrix(
    tasks=['passage-form', 'passage-content'],
    source_agents={'passage-content': 'qwen3.5-35b-a3b'},
    client=c,
)
print(f'Feature matrix: {features.shape[1]} cols, {len(features)} passages')

feat, groups = attach_extras(features, groups, c, include_lang=False)
print(f'After merge + embeddings: {len(feat)} passages')

In [ ]:
X = np.array(feat['embedding'].tolist(), dtype=np.float32)
print(f'Running UMAP on {X.shape}...')
coords = run_umap(X)
feat['umap_x'] = coords[:, 0]
feat['umap_y'] = coords[:, 1]
feat['abs_score'] = -feat['abstractness'].astype(float)  # + = more abstract
feat = add_period_bin(feat, year_col='year')

# Primary form-genre among top-tier form tags
FORM_TAGS = ['genre_novel', 'genre_romance', 'genre_allegory', 'genre_tale',
             'genre_biography', 'genre_novella', 'genre_dialogue', 'genre_satire']
available_tags = [t for t in FORM_TAGS if t in feat.columns]

def _primary_form(row):
    for t in available_tags:
        if row.get(t, 0) == 1:
            return t.replace('genre_', '')
    return '(other)'
feat['form_primary'] = feat.apply(_primary_form, axis=1)
print(feat['form_primary'].value_counts().head(10).to_string())

In [ ]:
def _first_col(suffixes):
    for suffix in suffixes:
        for col in feat.columns:
            if col.endswith(suffix):
                return col
    return None

plot_umap_overview(
    feat,
    form_tags=available_tags,
    annotation_cols=[
        _first_col(['uses_nominalization']),
        _first_col(['abstractions_as_agents']),
        _first_col(['concrete_bespeaks_abstract']),
    ],
    figname='umap_passages_overview.png',
    title=f'Passage-level UMAP of {len(feat):,} form-annotated passages',
)
plt.show()

## 2. Genre-faceted passage UMAP

Same points, one panel per top-tier form-genre, coloured by abstractness and by period.

In [ ]:
min_n = 20
top_forms = [g for g in available_tags if (feat[g] == 1).sum() >= min_n]
top_forms_clean = [g.replace('genre_', '') for g in top_forms]
print(f'Top forms (>= {min_n} passages): {top_forms_clean}')

# By abstractness
valid = feat['abs_score'].notna()
vmin, vmax = feat.loc[valid, 'abs_score'].quantile([0.05, 0.95])
norm = Normalize(vmin=vmin, vmax=vmax)
facet_by_genre(feat, top_forms, 'abs_score', norm=norm, cmap='RdBu',
               figname='umap_passages_genre_abstractness.png',
               title='Passage UMAP by form-genre, coloured by abstractness')
plt.show()

# By period
facet_by_genre(feat, top_forms, period_mode=True,
               figname='umap_passages_genre_period.png',
               title='Passage UMAP by form-genre, coloured by period')
plt.show()

## 3. Text-level UMAP (mean-pooled, all EN+FR)

Mean-pool all passage embeddings to one vector per text. Run two UMAPs: raw (language separates) and language-centered (cross-lingual structure).

In [ ]:
# All texts that have embeddings
print('Fetching all passage embeddings (this is the slowest step)...')
all_emb = c.query_df("SELECT _id, embedding FROM lltk.passage_embeddings WHERE scheme='p500'")
print(f'Loaded {len(all_emb)} passage embeddings')

text_ids_mp, X_text = mean_pool_to_text(all_emb)
print(f'Mean-pooled: {X_text.shape}')

text_df = pd.DataFrame({'_id': text_ids_mp})

genre_all = load_genre_extras(text_ids_mp, client=c)
text_df = text_df.merge(genre_all, left_on='_id', right_index=True, how='left')

meta_all = c.query_df('SELECT _id, year, lang FROM lltk.texts FINAL WHERE _id IN %(ids)s',
                      parameters={'ids': text_ids_mp}).drop_duplicates('_id')
text_df = text_df.merge(meta_all, on='_id', how='left')

# Text-level abstractness (already 1:1 per _id in abstraction.scores)
scores_text = c.query_df(
    "SELECT _id, `Abs-Conc.Median.median` AS abstractness FROM abstraction.scores WHERE _id IN %(ids)s",
    parameters={'ids': text_ids_mp},
).drop_duplicates('_id')
text_df = text_df.merge(scores_text, on='_id', how='left')
text_df['abs_score'] = -text_df['abstractness'].astype(float)
text_df = add_period_bin(text_df, year_col='year')
print(f'Text DF: {len(text_df)} rows, {text_df["lang"].value_counts().to_dict()}')

In [8]:
# Align text_df with X_text (drop dupes from merge)
text_df = text_df.drop_duplicates('_id').reset_index(drop=True)
# Re-key X to align
id_to_idx = {i: k for k, i in enumerate(text_ids_mp)}
X_text_aligned = np.array([X_text[id_to_idx[i]] for i in text_df['_id']], dtype=np.float32)
print(f'Aligned X: {X_text_aligned.shape}')

# UMAP raw
print('Running raw UMAP...')
coords_raw = run_umap(X_text_aligned)
text_df['umap_x_raw'] = coords_raw[:, 0]
text_df['umap_y_raw'] = coords_raw[:, 1]

# UMAP language-centered
print('Running language-centered UMAP...')
X_centered = center_by_group(X_text_aligned, text_df['lang'].fillna('?').tolist())
coords_c = run_umap(X_centered)
text_df['umap_x_c'] = coords_c[:, 0]
text_df['umap_y_c'] = coords_c[:, 1]

Aligned X: (3405, 1024)
Running raw UMAP...
Running language-centered UMAP...


In [ ]:
plot_text_umap_4panel(text_df, xcol='umap_x_raw', ycol='umap_y_raw',
                      form_tags=available_tags,
                      figname='umap_texts_raw_4panel.png',
                      title_suffix=' — raw embeddings')
plt.show()

plot_text_umap_4panel(text_df, xcol='umap_x_c', ycol='umap_y_c',
                      form_tags=available_tags,
                      figname='umap_texts_crosslingual_4panel.png',
                      title_suffix=' — language-centered')
plt.show()

## 4. Genre-faceted text UMAP (crosslingual space)

Top 20 tags from `load_genre_extras`, each faceted, colored by period and by abstractness.

In [ ]:
# Top 20 genre tags by text count
tag_counts = (genre_all > 0).sum().sort_values(ascending=False).head(20)
print(tag_counts)
top20_cols = tag_counts.index.tolist()

# Use the crosslingual coordinates for faceting
tfd = text_df.copy()
tfd['umap_x'] = tfd['umap_x_c']
tfd['umap_y'] = tfd['umap_y_c']

valid = tfd['abs_score'].notna()
vmin, vmax = tfd.loc[valid, 'abs_score'].quantile([0.05, 0.95])
norm = Normalize(vmin=vmin, vmax=vmax)
facet_by_genre(tfd, top20_cols, 'abs_score', norm=norm,
               figname='umap_texts_crosslingual_genre_abstractness.png',
               title='Text UMAP (cross-lingual) by genre, coloured by abstractness')
plt.show()

facet_by_genre(tfd, top20_cols, period_mode=True,
               figname='umap_texts_crosslingual_genre_period.png',
               title='Text UMAP (cross-lingual) by genre, coloured by period')
plt.show()

In [12]:
c.close()